# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "BioMistral/BioMistral-7B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading BioMistral model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("BioMistral loaded successfully!")

def generate_answer(question, max_new_tokens=120):
    messages = [
        {
            "role": "user",
            "content": (
                "Answer the following educational medical question "
                "clearly and cautiously. Do not invent evidence.\n\n"
                f"Question: {question}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )


question = (
    "What is the role of cholesterol and blood pressure "
    "in assessing cardiovascular risk?"
)

answer = generate_answer(question)

print("Question:", question)
print("\nBioMistral response:\n")
print(answer)



def generate_rag_answer(context, question, max_new_tokens=180):
    prompt = f"""
You are an assistant in a medical research and education system.

Answer the question using ONLY the medical evidence provided below.

Rules:
1. Do not add medical facts that are not supported by the evidence.
2. If the evidence does not contain the answer, explicitly say:
   "The provided evidence is insufficient to answer this question."
3. Do not present the answer as a diagnosis or treatment recommendation.

Retrieved medical evidence:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


medical_context = """
Source: Example educational passage about cardiovascular risk.

High blood pressure is a risk factor for cardiovascular disease.
Elevated LDL cholesterol can contribute to plaque accumulation
in arteries. Cardiovascular risk assessment considers multiple
factors, and these two factors alone do not establish a diagnosis.
"""

question = (
    "According to the provided evidence, how can high blood pressure "
    "and elevated LDL cholesterol affect cardiovascular risk?"
)

answer = generate_rag_answer(medical_context, question)

print("QUESTION:")
print(question)

print("\nRETRIEVED EVIDENCE:")
print(medical_context)

print("\nBIOMISTRAL ANSWER:")
print(answer)



unsupported_question = (
    "According to the provided evidence, what exact percentage "
    "of cardiovascular risk is caused by high blood pressure?"
)

unsupported_answer = generate_rag_answer(
    medical_context,
    unsupported_question
)

print("QUESTION:")
print(unsupported_question)

print("\nBIOMISTRAL ANSWER:")
print(unsupported_answer)



def generate_rag_answer_v2(context, question, max_new_tokens=220):
    prompt = f"""
You are a medical research assistant.

Answer the question using only the evidence provided.
Follow these instructions carefully:

1. Answer the exact question, not just the general topic.
2. Explain each relevant mechanism stated in the evidence.
3. Use two concise bullet points when two factors are asked about.
4. Do not introduce facts absent from the evidence.
5. If the evidence is insufficient, clearly state what is missing.
6. Do not make a diagnosis or recommend treatment.

EVIDENCE:
{context}

QUESTION:
{question}

EVIDENCE-BASED ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


improved_answer = generate_rag_answer_v2(
    medical_context,
    question
)

print("IMPROVED BIOMISTRAL ANSWER:")
print(improved_answer)



simple_prompt = """
Use the following evidence to answer the question in 3 short sentences.
Include every relevant fact from the evidence.
Do not add information that is not provided.

Evidence:
High blood pressure is a risk factor for cardiovascular disease.
Elevated LDL cholesterol can contribute to plaque accumulation
in arteries. Cardiovascular risk assessment considers multiple
factors, and these two factors alone do not establish a diagnosis.

Question:
How do high blood pressure and elevated LDL cholesterol affect
cardiovascular risk?

Answer:
"""

inputs = tokenizer(
    simple_prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

print(tokenizer.decode(new_tokens, skip_special_tokens=True))
print("\nGenerated tokens:", len(new_tokens))
print(
    "Finished because EOS token:",
    outputs[0, -1].item() == tokenizer.eos_token_id
)



test_context = """
Elevated LDL cholesterol can contribute to plaque
accumulation in arteries.
"""

test_question = (
    "According to the evidence, what can elevated LDL "
    "cholesterol contribute to?"
)

test_answer = generate_rag_answer_v2(
    test_context,
    test_question,
    max_new_tokens=120
)

print("QUESTION:", test_question)
print("\nANSWER:", test_answer)



test_cases = [
    {
        "name": "Test 1: Fact extraction",
        "context": (
            "Elevated LDL cholesterol can contribute to "
            "plaque accumulation in arteries."
        ),
        "question": (
            "What can elevated LDL cholesterol contribute to?"
        )
    },
    {
        "name": "Test 2: Multiple facts",
        "context": (
            "High blood pressure is a risk factor for "
            "cardiovascular disease. Elevated LDL cholesterol "
            "can contribute to plaque accumulation in arteries."
        ),
        "question": (
            "Explain the two cardiovascular risk factors "
            "mentioned in the evidence."
        )
    },
    {
        "name": "Test 3: Missing information",
        "context": (
            "High blood pressure is a risk factor for "
            "cardiovascular disease."
        ),
        "question": (
            "What exact percentage of cardiovascular risk "
            "is caused by high blood pressure according to "
            "the evidence?"
        )
    }
]

for case in test_cases:
    print("=" * 65)
    print(case["name"])
    print("Question:", case["question"])

    response = generate_rag_answer_v2(
        case["context"],
        case["question"],
        max_new_tokens=150
    )

    print("Answer:", response)
    print()



context = """
FACT 1: High blood pressure is a risk factor for
cardiovascular disease.

FACT 2: Elevated LDL cholesterol can contribute
to plaque accumulation in arteries.
"""

question = (
    "Explain both FACT 1 and FACT 2. You must address "
    "each fact separately. Do not omit either fact."
)

answer = generate_rag_answer_v2(
    context,
    question,
    max_new_tokens=180
)

print("ANSWER:")
print(answer)


def generate_rag_answer(question, retrieved_passages, max_new_tokens=250):
    context = ""

    for i, passage in enumerate(retrieved_passages, start=1):
        context += (
            f"\nSource {i}: {passage['source']}, "
            f"page {passage['page']}\n"
            f"{passage['text']}\n"
        )

    prompt = f"""
You are an educational clinical decision-support assistant.

Answer the user's question using only the medical evidence
provided below.

Rules:
1. Do not invent medical facts or citations.
2. If the evidence is insufficient, state that clearly.
3. Keep the answer clear and cautious.
4. Include source filenames and page numbers for the evidence used.
5. Do not claim to diagnose a patient.

Medical evidence:
{context}

Question:
{question}

Evidence-based answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

question = "What are common risk factors for heart disease?"

retrieved_passages = [
    {
        "source": "Heart attack - WHO.pdf",
        "page": 3,
        "text": (
            "Behavioural risk factors include tobacco use, "
            "a diet high in salt, sugar and unhealthy fats, "
            "and physical inactivity. Raised blood pressure, "
            "blood glucose and blood lipids also increase risk."
        )
    },
    {
        "source": "Heart Attack - Causes and Risk Factors _ NHLBI, NIH.pdf",
        "page": 4,
        "text": (
            "Risk factors include high blood triglycerides, "
            "overweight and obesity. Age also influences risk."
        )
    }
]

answer = generate_rag_answer(question, retrieved_passages)

print("Question:", question)
print("\nBioMistral RAG response:\n")
print(answer)

%pip install -q pinecone sentence-transformers

from google.colab import userdata
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# Connect to the existing Pinecone index
api_key = userdata.get("PINECONE_API_KEY")

pc = Pinecone(api_key=api_key)
index = pc.Index("heart-disease-knowledge")

# Load the exact embedding model used during indexing
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Pinecone connected.")
print("Embedding model loaded.")
print("Index statistics:", index.describe_index_stats())

query = "What are the common risk factors for heart disease?"

# Convert the question into a 384-dimensional vector
query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
).tolist()

# Search the existing Pinecone index
search_results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

print("Question:", query)
print("\nRetrieved medical evidence:\n")

for rank, match in enumerate(search_results["matches"], start=1):
    metadata = match["metadata"]

    print(f"Result {rank}")
    print(f"Similarity score: {match['score']:.4f}")
    print(f"Source: {metadata.get('source', 'Unknown')}")
    print(f"Page: {metadata.get('page', 'Unknown')}")
    print(f"Text: {metadata.get('text', 'No text available')[:700]}")
    print("-" * 80)

query = "What are the common risk factors for heart disease?"

# Convert the question into an embedding
query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
).tolist()

# Retrieve the five most relevant passages
search_results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

context_parts = []

for rank, match in enumerate(search_results["matches"], start=1):
    metadata = match["metadata"]

    text = metadata.get("text", "")
    source = metadata.get("source", "Unknown source")
    page = metadata.get("page", "Unknown page")

    if text.strip():
        context_parts.append(
            f"[Source {rank}: {source}, page {page}]\n{text}"
        )

context = "\n\n".join(context_parts)

print("Retrieved passages:", len(context_parts))
print("Context length:", len(context), "characters")
print("\nContext preview:\n")
print(context[:2500])

prompt = f"""
You are an AI assistant supporting a clinical
decision-support research project.

Answer the question using the medical evidence
provided in the context.

Instructions:
1. Use the supplied context as the main evidence.
2. Do not invent medical facts or unsupported details.
3. If the context does not contain enough information,
   clearly state that limitation.
4. Include source filenames and page numbers where relevant.
5. Do not claim to diagnose a patient.
6. Present the answer in clear, understandable language.

Medical evidence:
{context}

Question:
{query}

Answer:
"""

print(prompt[:1500])

Generate an answer using BioMistral and Pinecone evidence


inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=3500
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("QUESTION:")
print(query)

print("\nBIOMISTRAL RAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")
for rank, match in enumerate(search_results["matches"], start=1):
    metadata = match["metadata"]
    print(
        f"{rank}. {metadata.get('source', 'Unknown source')} "
        f"(Page {metadata.get('page', 'Unknown')})"
    )


Inspect the retrieved passages


for rank, match in enumerate(
    search_results["matches"], start=1
):
    metadata = match.get("metadata", {})

    print(f"\n{'=' * 70}")
    print(f"SOURCE {rank}")
    print(f"Filename: {metadata.get('source', 'Unknown')}")
    print(f"Page: {metadata.get('page', 'Unknown')}")
    print(f"Similarity score: {match.get('score', 'N/A')}")

    print("\nRetrieved passage:")
    print(
        metadata.get(
            "text",
            metadata.get("chunk_text", "Text field not found")
        )
    )

 Test RAG answer generation with multiple questions


test_questions = [
    "What are the common risk factors for heart disease?",
    "Which risk factors for heart disease can be modified?",
    "Which risk factors for heart disease cannot be modified?",
    "How can unhealthy lifestyle habits contribute to heart disease?"
]

def test_rag_question(question, top_k=5):
    query_embedding = embedding_model.encode(
      question,
      convert_to_numpy=True,
      normalize_embeddings=True
    ).tolist()

    search_results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    retrieved_passages = []

    for match in search_results["matches"]:
        metadata = match.get("metadata", {})
        passage = (
            metadata.get("text")
            or metadata.get("chunk_text")
            or metadata.get("content")
            or ""
        )

        if passage.strip():
            retrieved_passages.append(passage.strip())

    if not retrieved_passages:
        return {
            "question": question,
            "context": "",
            "answer": "No relevant passages were retrieved."
        }

    context = "\n\n".join(retrieved_passages)

    messages = [
        {
            "role": "user",
            "content": f"""Answer the question using the medical
context provided below.

If the context does not contain enough information, say so.
Do not invent medical facts.

Context:
{context}

Question: {question}

Answer:"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "context": context,
        "answer": answer
    }

rag_test_results = []

for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    result = test_rag_question(question)
    rag_test_results.append(result)

    print("\nBIOMISTRAL ANSWER:")
    print(result["answer"])






import pandas as pd

evaluation_df = pd.DataFrame([
    {
        "Question": result["question"],
        "Generated Answer": result["answer"]
    }
    for result in rag_test_results
])

evaluation_path = "/content/rag_evaluation_results.csv"

evaluation_df.to_csv(evaluation_path, index=False)

print("RAG evaluation results saved successfully.")
print("File:", evaluation_path)

display(evaluation_df)


NEED TO TEST CITATION PART

Generate an answer with citations

source_list = []

for rank, match in enumerate(
    search_results["matches"], start=1
):
    metadata = match.get("metadata", {})

    source = metadata.get("source", "Unknown source")
    page = metadata.get("page", "Unknown")

    source_list.append(
        f"[{rank}] {source}, page {page}"
    )

rag_prompt = f"""
You are a medical information assistant for a
clinical decision support research project.

Answer the user's question using ONLY the supplied
medical evidence.

Instructions:
1. Include only claims supported by the evidence.
2. Place a source citation such as [1] or [2]
   immediately after the supported claim.
3. Do not invent facts, statistics, or references.
4. If the evidence does not support a claim,
   do not include that claim.
5. Give a clear, concise answer.
6. Do not diagnose the user or prescribe treatment.

RETRIEVED MEDICAL EVIDENCE:
{context}

USER QUESTION:
{query}

ANSWER:
"""

inputs = tokenizer(
    rag_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=3500
).to(model.device)


with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][
    inputs["input_ids"].shape[-1]:
]

answer_with_citations = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

# Display the answer and source references
print("QUESTION:")
print(query)

print("\nBIOMISTRAL RAG ANSWER:")
print(answer_with_citations)

print("\nRETRIEVED SOURCES:")
for source in source_list:
    print(source)



# Step 9.7: Fix empty BioMistral responses

# Build a shorter context from the retrieved passages
evidence_parts = []

for rank, match in enumerate(
    search_results["matches"][:5], start=1
):
    metadata = match.get("metadata", {})

    # Adjust these field names if your Pinecone schema differs
    passage = (
        metadata.get("text")
        or metadata.get("chunk_text")
        or metadata.get("content")
    )

    if passage:
        evidence_parts.append(
            f"[{rank}] "
            f"{metadata.get('source', 'Unknown source')}, "
            f"page {metadata.get('page', 'Unknown')}:\n"
            f"{passage[:900]}"
        )

short_context = "\n\n".join(evidence_parts)

# Keep the instructions short and explicit
short_prompt = f"""<s>[INST]
Answer the question using the medical evidence below.
Give 3 concise bullet points.
Cite supporting evidence using [1], [2], [3], [4], or [5].
Do not invent facts or citations.

Evidence:
{short_context}

Question: {query}

Answer with the three bullet points only.
[/INST]"""

print("Prompt length:", len(short_prompt), "characters")
print("Evidence passages included:", len(evidence_parts))

# Tokenize
inputs = tokenizer(
    short_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2500
).to(model.device)

print("Input tokens:", inputs["input_ids"].shape[-1])

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Extract only the newly generated tokens
input_length = inputs["input_ids"].shape[-1]
new_tokens = outputs[0][input_length:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("\nGENERATED TOKEN COUNT:", new_tokens.shape[-1])
print("\nRAG ANSWER:")
print(answer if answer else "[Empty answer generated]")



# Step 9.8: Inspect BioMistral generation settings

print("MODEL:", MODEL_ID)

print("\nTOKENIZER CHAT TEMPLATE:")
print(tokenizer.chat_template)

print("\nGENERATION CONFIGURATION:")
print(model.generation_config)

print("\nSPECIAL TOKENS:")
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)
print("BOS token:", tokenizer.bos_token)
print("BOS token ID:", tokenizer.bos_token_id)


Test basic BioMistral generation

test_prompt = (
    "What are three common risk factors for heart disease? "
    "Answer in three short bullet points."
)

test_inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    test_outputs = model.generate(
        **test_inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

test_new_tokens = test_outputs[0][
    test_inputs["input_ids"].shape[-1]:
]

test_answer = tokenizer.decode(
    test_new_tokens,
    skip_special_tokens=True
).strip()

print("Generated token count:", test_new_tokens.shape[-1])
print("Test answer:", repr(test_answer))


messages = [
    {
        "role": "user",
        "content": (
            "What are three common risk factors for "
            "heart disease? Give three short bullet points."
        )
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", inputs["input_ids"].shape[-1])

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )


new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

test_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("\nGENERATED TOKEN COUNT:", new_tokens.shape[-1])
print("\nTEST ANSWER:")
print(test_answer if test_answer else "[Empty answer]")


BioMistral + Pinecone RAG

messages = [
    {
        "role": "user",
        "content": f"""
Answer the question using only the retrieved medical evidence.

Give 3 to 5 concise bullet points.
Cite supporting passages using their source numbers,
such as [1] or [2].
Do not invent medical claims or citations.

RETRIEVED EVIDENCE:
{context}

QUESTION:
{query}
"""
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("QUESTION:")
print(query)

print("\nBIOMISTRAL RAG ANSWER:")
print(answer if answer else "[Empty answer generated]")

print("\nRETRIEVED SOURCES:")
for rank, match in enumerate(
    search_results["matches"], start=1
):
    metadata = match.get("metadata", {})
    print(
        f"[{rank}] {metadata.get('source', 'Unknown source')}, "
        f"page {metadata.get('page', 'Unknown')}"
    )



# Step 9.12: Display the generated answer with source references

print("QUESTION:")
print(query)

print("\nBIOMISTRAL ANSWER:")
print(answer)

print("\nEVIDENCE SOURCES:")

for rank, match in enumerate(
    search_results["matches"], start=1
):
    metadata = match.get("metadata", {})

    source = metadata.get("source", "Unknown source")
    page = metadata.get("page", "Unknown")

    print(f"[{rank}] {source} — Page {page}")

print(
    "\nNote: The sources above were retrieved for this "
    "question. Each claim must be checked against the "
    "passages before its citation is considered verified."
)



# Step 9.13: Suggest supporting sources for answer claims

import re

# Extract source information and passage text
source_records = []

for rank, match in enumerate(
    search_results["matches"], start=1
):
    metadata = match.get("metadata", {})

    passage = (
        metadata.get("text")
        or metadata.get("chunk_text")
        or metadata.get("content")
        or ""
    )

    source_records.append({
        "id": rank,
        "source": metadata.get("source", "Unknown source"),
        "page": metadata.get("page", "Unknown"),
        "text": passage.lower()
    })

# Split the generated answer into sentences
claims = re.split(r"(?<=[.!?])\s+", answer.strip())

# Basic keyword matching for candidate evidence
keywords = {
    "high blood pressure": ["blood pressure", "hypertension"],
    "high cholesterol": ["cholesterol", "blood lipids"],
    "diabetes": ["diabetes", "blood glucose", "blood sugar"],
    "obesity": ["obesity", "overweight"],
    "smoking": ["smoking", "tobacco", "secondhand smoke"],
    "diet": ["diet", "sodium", "salt", "saturated fats", "sugar"],
    "physical inactivity": ["physical inactivity", "physically active"],
    "sleep": ["sleep"],
    "stress": ["stress"],
    "neighborhood access": [
        "healthy foods", "walking paths", "community centers",
        "gyms", "housing"
    ]
}

print("CLAIMS AND CANDIDATE SOURCES\n")

for claim in claims:
    if not claim.strip():
        continue

    claim_lower = claim.lower()
    matched_sources = []

    for record in source_records:
        matched_terms = []

        for topic, terms in keywords.items():
            if topic in claim_lower:
                matched_terms.extend(
                    term for term in terms
                    if term in record["text"]
                )

        if matched_terms:
            matched_sources.append(record)

    print("Claim:", claim)

    if matched_sources:
        for record in matched_sources:
            print(
                f"  Candidate: [{record['id']}] "
                f"{record['source']} — Page {record['page']}"
            )
    else:
        print("  Candidate sources: None identified by keyword matching")

    print()



# Step 9.14: Match individual claims to retrieved passages

risk_factors = {
    "High blood pressure": [
        "blood pressure", "hypertension"
    ],
    "High cholesterol": [
        "cholesterol", "blood lipids", "triglycerides"
    ],
    "Diabetes / high blood glucose": [
        "diabetes", "blood glucose", "blood sugar"
    ],
    "Overweight / obesity": [
        "obesity", "overweight"
    ],
    "Smoking / tobacco exposure": [
        "smoking", "tobacco", "secondhand smoke"
    ],
    "Unhealthy diet": [
        "sodium", "salt", "saturated fats",
        "unhealthy fats", "sugar", "diet"
    ],
    "Physical inactivity": [
        "physical inactivity", "physically active",
        "physical activity"
    ],
    "Insufficient sleep": [
        "sleep"
    ],
    "Stress": [
        "stress"
    ],
    "Limited neighborhood resources": [
        "healthy foods", "walking paths",
        "community centers", "gyms", "housing"
    ],
    "Age / family history": [
        "age", "family history"
    ],
    "Air pollution": [
        "air pollution"
    ]
}

# Extract passages from Pinecone results
source_records = []

for rank, match in enumerate(
    search_results["matches"], start=1
):
    metadata = match.get("metadata", {})

    passage = (
        metadata.get("text")
        or metadata.get("chunk_text")
        or metadata.get("content")
        or ""
    )

    source_records.append({
        "id": rank,
        "source": metadata.get("source", "Unknown source"),
        "page": metadata.get("page", "Unknown"),
        "text": passage.lower()
    })

# Find passages containing the terms for each risk factor
for factor, terms in risk_factors.items():
    print(f"\n{factor}:")

    found = False

    for record in source_records:
        matched_terms = [
            term for term in terms
            if term in record["text"]
        ]

        if matched_terms:
            found = True
            print(
                f"  [{record['id']}] {record['source']} "
                f"(Page {record['page']})"
            )
            print(
                "      Matching terms:",
                ", ".join(matched_terms)
            )

    if not found:
        print("  No keyword match in retrieved passages.")



# Step 9.15: Inspect evidence passages for each risk factor

risk_factors = {
    "High blood pressure": ["blood pressure", "hypertension"],
    "High cholesterol": ["cholesterol", "blood lipids", "triglycerides"],
    "Diabetes / high blood glucose": ["diabetes", "blood glucose", "blood sugar"],
    "Overweight / obesity": ["obesity", "overweight"],
    "Smoking / tobacco exposure": ["smoking", "tobacco", "secondhand smoke"],
    "Unhealthy diet": ["sodium", "salt", "saturated fats", "unhealthy fats", "sugar", "diet"],
    "Physical inactivity": ["physical inactivity", "physically active", "physical activity"],
    "Insufficient sleep": ["sleep"],
    "Stress": ["stress"],
    "Limited neighborhood resources": [
        "healthy foods", "walking paths", "community centers",
        "gyms", "housing"
    ],
    "Age / family history": ["age", "family history"],
    "Air pollution": ["air pollution"]
}

# Extract passages from the existing Pinecone search results
source_records = []

for rank, match in enumerate(search_results["matches"], start=1):
    metadata = match.get("metadata", {})

    passage = (
        metadata.get("text")
        or metadata.get("chunk_text")
        or metadata.get("content")
        or ""
    )

    source_records.append({
        "id": rank,
        "source": metadata.get("source", "Unknown source"),
        "page": metadata.get("page", "Unknown"),
        "text": passage
    })

# Print the evidence passages for manual verification
for factor, terms in risk_factors.items():
    print("\n" + "=" * 70)
    print(f"RISK FACTOR: {factor}")
    print("=" * 70)

    found = False

    for record in source_records:
        passage_lower = record["text"].lower()

        matched_terms = [
            term for term in terms
            if term in passage_lower
        ]

        if matched_terms:
            found = True

            print(
                f"\nSource [{record['id']}]: "
                f"{record['source']}"
            )
            print(f"Page: {record['page']}")
            print("Matched terms:", ", ".join(matched_terms))
            print("Evidence passage:")
            print(record["text"])

    if not found:
        print("No matching evidence passage was found.")
